# Iteration 1: Brute-Force KNN Search

Exact KNN via matrix multiplication on CUDA (RTX 3080).
Cosine similarity = dot product on L2-normalised vectors.

In [10]:
!pip install -q torch psutil numpy

In [11]:
import sys, os, time
import psutil
import numpy as np
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Device: cpu


In [12]:
TOP_K   = 5
N_SYNTH = 500_000
DIM     = 256

## Load corpus

Run `python scripts/build_storage.py` first — reads `data/wikipedia_embeddings_256d.npy` and writes the store files.

In [13]:
try:
    from storage import Float32Store

    store = Float32Store()
    store.load()   # reads storage_data/store_float32.npy from disk
    corpus_np = store.data.astype(np.float32)
    N, D = corpus_np.shape
    print(f"Float32Store: {N:,} × {D} | {store.get_memory_footprint():.1f} MB")

except Exception as e:
    # fallback to random vectors for development / testing without real data
    print(f"Float32Store not available ({e}) — using synthetic data")
    rng = np.random.default_rng(42)
    corpus_np = rng.standard_normal((N_SYNTH, DIM)).astype(np.float32)
    N, D = corpus_np.shape

# L2-normalise once upfront so that dot product == cosine similarity at search time
corpus_np /= np.linalg.norm(corpus_np, axis=1, keepdims=True)

[Float32Store] Loading from /Users/dilfd/Documents/dls_project/local-wiki-rag/storage_data/store_float32.npy...
[Float32Store] Loaded. Shape: (500000, 256)
Float32Store: 500,000 × 256 | 488.3 MB


## Transfer corpus to GPU

In [14]:
# 500k × 256 × 4 bytes ≈ 500 MB — well within RTX 3080 VRAM (10 GB)
corpus_gpu = torch.from_numpy(corpus_np).to(device)

if device == "cuda":
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")

## BruteForceSearch

In [15]:
class BruteForceSearch:
    def __init__(self, corpus: torch.Tensor, device: str):
        self.corpus = corpus   # (N, D), L2-normalised, stored on GPU
        self.device = device

    def search(self, query: np.ndarray, top_k: int = TOP_K):
        q = torch.from_numpy(query.astype(np.float32)).to(self.device)
        q = F.normalize(q, dim=0)   # L2-normalise the query vector

        # single matmul covers the entire corpus: (N, D) @ (D,) → (N,)
        scores = self.corpus @ q

        top_scores, top_idx = torch.topk(scores, k=top_k)
        return top_idx.cpu().numpy(), top_scores.cpu().numpy()


brute = BruteForceSearch(corpus_gpu, device)
print(f"BruteForceSearch ready | {N:,} × {D}")

BruteForceSearch ready | 500,000 × 256


## Demo search

In [16]:
# warm-up: first CUDA call is slow due to kernel compilation, don't count it
_ = brute.search(np.random.randn(D).astype(np.float32))
if device == "cuda":
    torch.cuda.synchronize()

query_vec = np.random.randn(D).astype(np.float32)

t0 = time.perf_counter()
indices, scores = brute.search(query_vec, top_k=TOP_K)
if device == "cuda":
    torch.cuda.synchronize()   # wait for GPU to finish before stopping the clock
latency_ms = (time.perf_counter() - t0) * 1000

print(f"Latency: {latency_ms:.2f} ms")
print(f"\nTop-{TOP_K} results:")
print(f"{'Rank':>4}  {'Doc ID':>8}  {'Cosine Score':>12}")
print("-" * 30)
for rank, (idx, score) in enumerate(zip(indices, scores), 1):
    print(f"{rank:>4}  {idx:>8}  {score:>12.4f}")

Latency: 14.12 ms

Top-5 results:
Rank    Doc ID  Cosine Score
------------------------------
   1     63089        0.1817
   2    423983        0.1704
   3    159589        0.1672
   4    442415        0.1620
   5    190663        0.1604


## Memory usage

In [17]:
proc = psutil.Process(os.getpid())
print(f"Process RAM : {proc.memory_info().rss / 1024**2:.1f} MB")
print(f"Corpus size : {corpus_np.nbytes / 1024**2:.1f} MB  (float32, {N:,} × {D})")
if device == "cuda":
    print(f"VRAM alloc  : {torch.cuda.memory_allocated() / 1024**2:.1f} MB")

Process RAM : 1199.6 MB
Corpus size : 488.3 MB  (float32, 500,000 × 256)
